In [1]:
# 配置环境变量
from dotenv import load_dotenv
from langchain_core.messages import trim_messages
from openai import conversations
from pyexpat.errors import messages

load_dotenv()

import os
api_key = os.getenv("DASHSCOPE_API_KEY")
base_url = os.getenv("DASHSCOPE_BASE_URL")
if not api_key or api_key == "your_dashcope_api_key_here":
    raise ValueError("Please set your Dashscope API key in .env file.")
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain.tools import tool
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware
from langchain.agents.middleware import AgentMiddleware

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key or groq_api_key == "your_grouq_api_key_here":
    raise ValueError("Please set your Group API key in .env file.")
grop_model = init_chat_model(
    model="groq:llama-3.3-70b-versatile",
    api_key=groq_api_key,
)
model = init_chat_model(
    model="qwen-max",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url
)


# 统计模型调用次数

In [2]:
# 计数中间件
class CallCounterMiddleware(AgentMiddleware):
    """
    统计模型调用次数

    在中间件内部维护计数器（简单版本）
    """

    def __init__(self):
        super().__init__()
        self.count = 0  # 简单计数器

    def after_model(self, state, runtime):
        """模型响应后，增加计数"""
        self.count += 1
        print(f"\n[计数器] 模型调用次数: {self.count}")
        return None  # 不修改 state

# 最大调用限制

In [3]:
# 最大调用限制中间件
class MaxCallsMiddleware(AgentMiddleware):
    """
    通过抛出异常来阻止模型调用（更可靠的方式）
    """

    def __init__(self, max_calls=3):
        super().__init__()
        self.max_calls = max_calls
        self.count = 0  # 简单计数器

    def before_model(self, state, runtime):
        """检查调用次数，超过限制则抛出异常"""
        if self.count >= self.max_calls:
            print(f"\n[限制] 已达到最大调用次数 {self.max_calls}，停止调用")
            # 抛出自定义异常来阻止继续执行
            raise ValueError(f"已达到最大调用次数限制: {self.max_calls}")

        print(f"[限制] 当前调用次数: {self.count}/{self.max_calls}")
        return None

    def after_model(self, state, runtime):
        """增加计数"""
        self.count += 1
        print("次数+1")
        return None

# 消息修剪

In [14]:
# 消息修剪中间件
class MessageTrimmerMiddleware(AgentMiddleware):
    """
    限制消息数量

    before_model：修改消息队列
    需配合无 checkpointer 使用，否则历史会被恢复
    """

    def __init__(self, max_messages=5):
        super().__init__()
        self.max_messages = max_messages
        self.trimmed_count = 0 # 统计修剪次数

    def before_model(self, state, runtime):
        """模型调用前，修剪消息"""
        messages = state.get("messages", [])

        if len(messages) > self.max_messages:
            # 保留最近的 N 条消息
            trimmed_messages = messages[-self.max_messages:]
            self.trimmed_count += 1
            print(f"[修剪]消息从 {len(messages)} 条减少到 {len(trimmed_messages)}次修剪")
            return {"messages": trimmed_messages}

        return None

# 输出验证

In [15]:
# 输出验证中间件
class OutputValidationMiddleware(AgentMiddleware):
    """
    检查响应长度
    """

    def __init__(self, max_length=100):
        super().__init__()
        self.max_length = max_length

    def after_model(self, state, runtime):
        """模型调用后，验证输出"""
        messages = state.get("messages", [])
        if not messages:
            return None

        last_message = messages[-1]
        content = getattr(last_message, "content", '')

        if len(content) > self.max_length:
            print(f"[警告] 响应过长，长度为 {len(content)} 字符，已截断到 {self.max_length}")

        return None

# 日志中间件

In [25]:
# 日志中间件
class LoggingMiddleware(AgentMiddleware):
    """
    记录每次模型调用

    before_model：模型调用前执行
    after_model：模型调用后执行
    """

    def before_model(self, state, runtime):
        """模型调用前"""
        print("\n[日志] before_model：准备调用模型")
        print(f"[日志] 当前消息数：{len(state.get('messages', []))}")
        return None # 返回 None 表示继续正常流程

    def after_model(self, state, runtime):
        """模型调用后"""
        print("\n[日志] after_model：模型已响应")
        last_message = state.get("messages", [])[-1]
        print(f"[日志] 响应类型：{last_message.__class__.__name__}")
        return None # 返回 None 表示不修改状态

In [32]:
def example():
    db_path = "middleware_basics.sqlite"
    with SqliteSaver.from_conn_string(db_path) as checkpointer:
        agent = create_agent(
            model=model,
            checkpointer=checkpointer,
            system_prompt="""
                你是一个客服助手，
                能够记住用户的对话
            """,
            middleware=[
                # 模型调用计数
                CallCounterMiddleware(),
                # 模型最大调用限制
                MaxCallsMiddleware(max_calls=5),
                # 消息修剪
                # MessageTrimmerMiddleware(max_messages=4),
                # 输出验证
                OutputValidationMiddleware(max_length=50),
                # 自动摘要
                SummarizationMiddleware(
                    model=grop_model,
                    trigger=("tokens", 800)
                ),
                # PII 脱敏（隐私数据）
                # PIIMiddleware("email", strategy="redact"), #邮箱占位符替换
                # PIIMiddleware("phone", strategy="block"),  #电话拦截抛异常
                # PIIMiddleware("id_card", strategy="mask"),  #身份证部分掩码
                # 日志记录
                LoggingMiddleware()
            ]
        )

        config = {"configurable": {"thread_id": "test_user"}}
        conversations = [
            "你好",
            "我叫张三，今年18岁",
            "我叫什么名字？",
            "我多少岁来着",
        ]
        for conversation in conversations:
            print(f"用户：{conversation}")
            response = agent.invoke(
                {"messages": [
                    {"role": "user", "content": conversation},
                ]},
                config=config
            )
            print(f"助手：{response['messages'][-1].content}")



In [33]:
def main():
    print("\n"+"="*70)
    print("middleware_basics")
    print("="*70)

    try:
        example()
    except Exception as e:
        print(f"错误：{e}")

In [34]:
if __name__ == "__main__":
    main()


middleware_basics
用户：你好
[限制] 当前调用次数: 0/5

[日志] before_model：准备调用模型
[日志] 当前消息数：9

[日志] after_model：模型已响应
[日志] 响应类型：AIMessage
次数+1

[计数器] 模型调用次数: 1
助手：你好！有什么我可以帮助你的吗？
用户：我叫张三，今年18岁
[限制] 当前调用次数: 1/5

[日志] before_model：准备调用模型
[日志] 当前消息数：11

[日志] after_model：模型已响应
[日志] 响应类型：AIMessage
次数+1

[计数器] 模型调用次数: 2
助手：你好，张三！很高兴认识你。18岁正是青春洋溢的年纪呢。有什么我可以帮助你的吗？或者你想聊些什么？
用户：我叫什么名字？
[限制] 当前调用次数: 2/5

[日志] before_model：准备调用模型
[日志] 当前消息数：13

[日志] after_model：模型已响应
[日志] 响应类型：AIMessage
次数+1

[计数器] 模型调用次数: 3
助手：你叫张三。有什么其他问题或者需要帮助的地方吗？
用户：我多少岁来着
[限制] 当前调用次数: 3/5

[日志] before_model：准备调用模型
[日志] 当前消息数：15

[日志] after_model：模型已响应
[日志] 响应类型：AIMessage
次数+1

[计数器] 模型调用次数: 4
助手：你今年18岁。还有其他问题或者需要聊些什么吗？
